In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('/home/admin/Documents/98_model/src')
sys.path.append('/data01/Documents/98_model/src')

## Import

In [3]:
from project_paths import PROJECT_PATH, DATA_PATH, SRC_PATH
from target_columns.cols_small_y import (
    SMALL_Y_DEFECT_ELECTRODE,
    SMALL_Y_CTQ_ELECTRODE,
    SMALL_Y_DEFECT_WINDING,
    SMALL_Y_CTQ_WINDING,
    SMALL_Y_DEFECT_ASSEMBLY,
    SMALL_Y_CTQ_ASSEMBLY,
    SMALL_Y_DEFECT_WASHING,
    SMALL_Y_CTQ_ACTIVATION
)
import pandas as pd
from data_loader.data_loader import DataLoader
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score, mean_absolute_error
from tqdm import tqdm
import re
import warnings
import copy
from tqdm import trange
import gc
warnings.filterwarnings('ignore')

In [4]:
import polars as pl

## Load data by version

In [5]:
def tmp(x) :
    try : 
        y = x[0]
    except : 
        y = x
    return y

In [6]:
idx_len = 8

In [7]:
data = None

In [8]:
idx = 0
filename = f'feature_store_v10_n32s_{idx}.parquet'
data_chunk = pd.read_parquet(filename)


for col in [x for x in data_chunk.columns if 'Date' in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))

for col in [x for x in data_chunk.columns if 'ID' in x and 'Degas' not in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))
display(data_chunk)

if data is None : 
    data = data_chunk
else :
    data = pd.concat([data, data_chunk], axis=0)
    #data = pd.concat([data, data_chunk], axis=0, ignore_index=True, sort=False)
    del data_chunk
    gc.collect()

datetime64[ns]
object
object
object
object
datetime64[ns]
datetime64[ns]
float64
float64
object
object
object
object
object
object
object
object
object
object
object
object
object
object
object
float64
float64
float64
float64
object
object


,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,07TCED7LGC0021G2B2016718,59JFB112A1,5A2F201001,M2EMIX01602,2026-02-02 01:28:10,5CF1T1A5C1,M2ECOT001,2026-01-30 06:12:56,5CF1T1A5R2,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07TCED7LGC0021G2E2063322,59JFB142A1,5A2F201001,M2EMIX01602,2026-02-02 01:28:10,5AF2A131C1,M2ECOT002,2026-02-10 14:14:12,5AF2A131R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,07TCED7LGC0021G2G2018702,59JFB162A1,5A3F209041,M2EMIX01702,2026-02-09 19:11:46,5CF1V165C1,M2ECOT001,2026-01-31 18:56:59,5CF1V165R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07TCED7LGC0021G382040158,59JFC082A1,5C1F225029,M2EMIX01103,2026-02-26 00:00:57,5CF2R151C1,M2ECOT001,2026-02-27 17:44:29,5CF2R151R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07TCED7LGC0021G3S2044730,59JFC252A1,5C2F309007,M2EMIX01203,2026-03-09 21:02:14,5CF3E183C1,M2ECOT001,2026-03-14 20:58:31,5CF3E183R1,M2EROL015,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14824,07TCED7LGC0021G362009096,59JFC062A1,5C1F225029,M2EMIX01103,2026-02-26 00:00:57,5CF2R164C1,M2ECOT001,2026-02-27 20:08:34,5CF2R164R1,M2EROL012,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14825,07TCED7LGC0021G3H2001157,59JFC172A1,5C3F309003,M2EMIX01303,2026-03-09 12:48:18,5CF3B163C1,M2ECOT001,2026-03-11 17:22:16,5CF3B163R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14826,07TCED7LGC0021G102075671,59JFA312A1,5A1F105002,M2EMIX01502,2026-01-05 22:48:51,5AF1N176C1,M2ECOT002,2026-01-23 23:04:13,5AF1N176R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14827,07TCED7LGC0021G3H2101246,59JFC172A1,5A1F303001,M2EMIX01502,2026-03-03 17:45:32,5AF36126C1,M2ECOT002,2026-03-06 10:39:09,5AF36126R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
idx = 1
filename = f'feature_store_v10_n32s_{idx}.parquet'
data_chunk = pd.read_parquet(filename)


for col in [x for x in data_chunk.columns if 'Date' in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))

for col in [x for x in data_chunk.columns if 'ID' in x and 'Degas' not in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))
display(data_chunk)

if data is None : 
    data = data_chunk
else :
    data = pd.concat([data, data_chunk], axis=0)
    #data = pd.concat([data, data_chunk], axis=0, ignore_index=True, sort=False)
    del data_chunk
    gc.collect()

datetime64[ns]
object
object
object
object
datetime64[ns]
datetime64[ns]
float64
float64
object
object
object
object
object
object
object
object
object
object
object
object
object
object
object
float64
float64
float64
float64
object
object


,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,07TCED7LGC0021G3N2022899,59JFC222A1,5A2F316031,M2EMIX01602,2026-03-16 14:48:06,5AF3H133C1,M2ECOT002,2026-03-17 14:17:26,5AF3H133R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07TCED7LGC0021G3P2121173,59JFC232A1,5C2F309007,M2EMIX01203,2026-03-09 21:02:14,5CF3D1D3C1,M2ECOT001,2026-03-14 04:43:59,5CF3D1D3R1,M2EROL015,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,07TCED7LGC0021G272076469,59JFB072A1,5A2F201001,M2EMIX01602,2026-02-02 01:28:10,5AF23182C1,M2ECOT002,2026-02-04 01:10:41,5AF23182R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07TCED7LGC0021G2T2064167,59JFB262A1,5C1F213008,M2EMIX01103,2026-02-13 19:13:40,5CF2D185C1,M2ECOT001,2026-02-14 01:49:11,5CF2D185R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07TCED7LGC0021G292129951,59JFB092A1,5A2F201001,M2EMIX01602,2026-02-02 01:28:10,5CF1T1A2C1,M2ECOT001,2026-01-30 04:52:57,5CF1T1A2R2,M2EROL012,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14825,07TCED7LGC0021G2M2005584,59JFB212A1,5A2F201001,M2EMIX01602,2026-02-02 01:28:10,5AF2B117C1,M2ECOT002,2026-02-11 08:53:54,5CF2D184R1,M2EROL012,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14826,07TCED7LGC0021G3G2057229,59JFC162A1,5C3F309002,M2EMIX01303,2026-03-09 10:16:52,5CF3A185C1,M2ECOT001,2026-03-11 03:44:03,5CF3A185R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14827,07TCED7LGC0021G3V2043668,59JFC272A1,5A2F319052,M2EMIX01602,2026-03-19 06:47:09,5AF3J135C1,M2ECOT002,2026-03-19 13:41:56,5AF3J135R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14828,07TCED7LGC0021G2C2068476,59JFB122A1,5A2F201001,M2EMIX01602,2026-02-02 01:28:10,5AF29112C1,M2ECOT002,2026-02-09 14:55:30,5AF29112R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
idx = 2
filename = f'feature_store_v10_n32s_{idx}.parquet'
data_chunk = pd.read_parquet(filename)


for col in [x for x in data_chunk.columns if 'Date' in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))

for col in [x for x in data_chunk.columns if 'ID' in x and 'Degas' not in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))
display(data_chunk)

if data is None : 
    data = data_chunk
else :
    data = pd.concat([data, data_chunk], axis=0)
    #data = pd.concat([data, data_chunk], axis=0, ignore_index=True, sort=False)
    del data_chunk
    gc.collect()

datetime64[ns]
object
object
object
object
datetime64[ns]
datetime64[ns]
float64
float64
object
object
object
object
object
object
object
object
object
object
object
object
object
object
object
float64
float64
float64
float64
object
object


,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,07TCED7LGC0021G2S2052203,59JFB252A1,5C1F213008,M2EMIX01103,2026-02-13 19:13:40,5CF2D181C1,M2ECOT001,2026-02-14 00:20:12,5CF2D181R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07TCED7LGC0021G3R2015802,59JFC242A1,5C3F309003,M2EMIX01303,2026-03-09 12:48:18,5CF3B155C1,M2ECOT001,2026-03-11 16:11:13,5CF3B155R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,07TCED7LGC0021G3E2060814,59JFC142A1,5C3F309003,M2EMIX01303,2026-03-09 12:48:18,5CF3B161C1,M2ECOT001,2026-03-11 16:46:46,5CF3B161R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07TCED7LGC0021G3T2124126,59JFC262A1,5C2F309007,M2EMIX01203,2026-03-09 21:02:14,5CF3D153C1,M2ECOT001,2026-03-13 14:31:15,5CF3D153R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07TCED7LGC0021G342062946,59JFC042A1,5A2F201001,M2EMIX01602,2026-02-02 01:28:10,5AF22177C1,M2ECOT002,2026-02-03 00:37:57,5AF22177R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14824,07TCED7LGC0021G2D2055593,59JFB132A1,5C2F112017,M2EMIX01203,2026-01-12 10:09:37,5CF1H131C1,M2ECOT001,2026-01-17 12:24:57,5CF1H131R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14825,07TCED7LGC0021G2J2058338,59JFB182A1,5A2F201001,M2EMIX01602,2026-02-02 01:28:10,5AF23146C1,M2ECOT002,2026-02-03 16:41:20,5AF23146R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14826,07TCED7LGC0021G3T2068460,59JFC262A1,5A2F318044,M2EMIX01602,2026-03-18 07:28:19,5AF3I135C1,M2ECOT002,2026-03-18 13:17:01,5AF3I135R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14827,07TCED7LGC0021G3J2077742,59JFC182A1,5C3F309004,M2EMIX01303,2026-03-09 17:34:33,5CF3C152C1,M2ECOT001,2026-03-12 20:06:21,5CF3C152R1,M2EROL013,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
idx = 3
filename = f'feature_store_v10_n32s_{idx}.parquet'
data_chunk = pd.read_parquet(filename)


for col in [x for x in data_chunk.columns if 'Date' in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))

for col in [x for x in data_chunk.columns if 'ID' in x and 'Degas' not in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))
display(data_chunk)

if data is None : 
    data = data_chunk
else :
    data = pd.concat([data, data_chunk], axis=0)
    #data = pd.concat([data, data_chunk], axis=0, ignore_index=True, sort=False)
    del data_chunk
    gc.collect()

datetime64[ns]
object
object
object
object
datetime64[ns]
datetime64[ns]
float64
float64
object
object
object
object
object
object
object
object
object
object
object
object
object
object
object
float64
float64
float64
float64
object
object


,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,07TCED7LGC0021G2E2089051,59JFB142A1,5A2F201001,M2EMIX01602,2026-02-02 01:28:10,5CF1U155C1,M2ECOT001,2026-01-30 16:47:35,5CF1U155R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07TCED7LGC0021G3S2046030,59JFC252A1,5A2F318044,M2EMIX01602,2026-03-18 07:28:19,5AF3I167C1,M2ECOT002,2026-03-18 21:02:34,5AF3I167R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,07TCED7LGC0021G242031953,59JFB042A1,5A1F105002,M2EMIX01502,2026-01-05 22:48:51,5CF1S111C1,M2ECOT001,2026-01-28 07:15:23,5CF1S111R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07TCED7LGC0021G272064942,59JFB072A1,5A2F201001,M2EMIX01602,2026-02-02 01:28:10,5CF1T171C1,M2ECOT001,2026-01-29 20:27:22,5CF1T171R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07TCED7LGC0021G232061062,59JFB032A1,5A1F105002,M2EMIX01502,2026-01-05 22:48:51,5AF1L181C1,M2ECOT002,2026-01-22 04:30:40,5AF1L181R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14825,None,59JFC302A1,5C1F325056,M2EMIX01103,2026-03-25 10:17:47,5CF3P182C1,M2ECOT001,2026-03-25 22:57:45,5CF3P182R1,M2EROL012,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14826,None,59JFC302A1,5A4F320066,M2EMIX01802,2026-03-20 15:59:56,5AF3K175C1,M2ECOT002,2026-03-20 23:16:06,5AF3K175R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14827,None,59JFC302A1,5A4F320065,M2EMIX01802,2026-03-20 13:30:12,5AF3K141C1,M2ECOT002,2026-03-20 14:53:16,5AF3K141R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14828,None,59JFC302A1,5C2F309007,M2EMIX01203,2026-03-09 21:02:14,5CF3E168C1,M2ECOT001,2026-03-14 18:34:00,5CF3E168R1,M2EROL013,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
idx = 4
filename = f'feature_store_v10_n32s_{idx}.parquet'
data_chunk = pd.read_parquet(filename)


for col in [x for x in data_chunk.columns if 'Date' in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))

for col in [x for x in data_chunk.columns if 'ID' in x and 'Degas' not in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))
display(data_chunk)

if data is None : 
    data = data_chunk
else :
    data = pd.concat([data, data_chunk], axis=0)
    #data = pd.concat([data, data_chunk], axis=0, ignore_index=True, sort=False)
    del data_chunk
    gc.collect()

datetime64[ns]
object
object
object
object
datetime64[ns]
datetime64[ns]
float64
float64
object
object
object
object
object
object
object
object
object
object
object
object
object
object
object
float64
float64
float64
float64
object
object


,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,None,59JFC302A1,5C2F309007,M2EMIX01203,2026-03-09 21:02:14,5CF3E168C1,M2ECOT001,2026-03-14 18:34:00,5CF3E168R1,M2EROL013,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,None,59JFC302A1,5A4F320064,M2EMIX01802,2026-03-20 06:51:07,5AF3K141C1,M2ECOT002,2026-03-20 14:53:16,5AF3K141R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,None,59JFC302A1,5C1F324051,M2EMIX01103,2026-03-24 15:31:54,5CF3P182C1,M2ECOT001,2026-03-25 22:57:45,5CF3P182R1,M2EROL012,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,None,59JFC302A1,5C1F325056,M2EMIX01103,2026-03-25 10:17:47,5CF3P182C1,M2ECOT001,2026-03-25 22:57:45,5CF3P182R1,M2EROL012,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,None,59JFC302A1,5A4F320064,M2EMIX01802,2026-03-20 06:51:07,5AF3K177C1,M2ECOT002,2026-03-20 23:44:37,5AF3K177R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14824,None,59JFC302A1,5C2F324047,M2EMIX01203,2026-03-24 12:58:56,5CF3P135C1,M2ECOT001,2026-03-25 14:02:37,5CF3P135R1,M2EROL015,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14825,None,59JFC302A1,5C2F324047,M2EMIX01203,2026-03-24 12:58:56,5CF3P144C1,M2ECOT001,2026-03-25 15:15:50,5CF3P144R1,M2EROL012,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14826,None,59JFC302A1,5A4F320064,M2EMIX01802,2026-03-20 06:51:07,5AF3K141C1,M2ECOT002,2026-03-20 14:53:16,5AF3K141R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14827,None,59JFC302A1,5A4F320064,M2EMIX01802,2026-03-20 06:51:07,5AF3K167C1,M2ECOT002,2026-03-20 21:21:56,5AF3K167R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
idx = 5
filename = f'feature_store_v10_n32s_{idx}.parquet'
data_chunk = pd.read_parquet(filename)


for col in [x for x in data_chunk.columns if 'Date' in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))

for col in [x for x in data_chunk.columns if 'ID' in x and 'Degas' not in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))
display(data_chunk)

if data is None : 
    data = data_chunk
else :
    data = pd.concat([data, data_chunk], axis=0)
    #data = pd.concat([data, data_chunk], axis=0, ignore_index=True, sort=False)
    del data_chunk
    gc.collect()

datetime64[ns]
object
object
object
object
datetime64[ns]
datetime64[ns]
float64
float64
object
object
object
object
object
object
object
object
object
object
object
object
object
object
object
float64
float64
float64
float64
object
object


,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,None,59JFC062A1,5A2F201001,M2EMIX01602,2026-02-02 01:28:10,5AF23183C1,M2ECOT002,2026-02-04 01:38:38,5AF23183R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,None,59JFC302A1,5A2F318044,M2EMIX01602,2026-03-18 07:28:19,5AF3I186C1,M2ECOT002,2026-03-19 01:24:29,5AF3I186R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,None,59JFC302A1,5A2F319052,M2EMIX01602,2026-03-19 06:47:09,5AF3J185C1,M2ECOT002,2026-03-20 01:20:13,5AF3J185R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,None,59JFC302A1,5C1F324051,M2EMIX01103,2026-03-24 15:31:54,5CF3P183C1,M2ECOT001,2026-03-25 23:33:16,5CF3P183R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,None,59JFC302A1,5A4F320064,M2EMIX01802,2026-03-20 06:51:07,5AF3K141C1,M2ECOT002,2026-03-20 14:53:16,5AF3K141R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14825,None,59JFC302A1,5A2F319052,M2EMIX01602,2026-03-19 06:47:09,5AF3J118C1,M2ECOT002,2026-03-19 09:20:12,5AF3J118R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14826,None,59JFC302A1,5A4F320066,M2EMIX01802,2026-03-20 15:59:56,5AF3K177C1,M2ECOT002,2026-03-20 23:44:37,5AF3K177R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14827,None,59JFC302A1,5A2F318044,M2EMIX01602,2026-03-18 07:28:19,5AF3I186C1,M2ECOT002,2026-03-19 01:24:29,5AF3I186R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14828,None,59JFC302A1,5C2F309007,M2EMIX01203,2026-03-09 21:02:14,5CF3E166C1,M2ECOT001,2026-03-14 17:58:29,5CF3E166R1,M2EROL013,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
idx = 6
filename = f'feature_store_v10_n32s_{idx}.parquet'
data_chunk = pd.read_parquet(filename)


for col in [x for x in data_chunk.columns if 'Date' in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))

for col in [x for x in data_chunk.columns if 'ID' in x and 'Degas' not in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))
display(data_chunk)

if data is None : 
    data = data_chunk
else :
    data = pd.concat([data, data_chunk], axis=0)
    #data = pd.concat([data, data_chunk], axis=0, ignore_index=True, sort=False)
    del data_chunk
    gc.collect()

object
object
object
object
object
datetime64[ns]
datetime64[ns]
float64
object
object
object
object
object
object
object
object
object
object
object
object
object
object
object
float64
float64
float64
float64
object
object


,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,None,59JFC302A1,5A4F320064,M2EMIX01802,2026-03-20 06:51:07,5AF3K141C1,M2ECOT002,2026-03-20 14:53:16,5AF3K141R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,None,59JFC302A1,5C2F324047,M2EMIX01203,2026-03-24 12:58:56,5CF3P154C1,M2ECOT001,2026-03-25 17:37:57,5CF3P154R1,M2EROL012,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,None,59JFC302A1,5C2F324047,M2EMIX01203,2026-03-24 12:58:56,5CF3P144C1,M2ECOT001,2026-03-25 15:15:50,5CF3P144R1,M2EROL012,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,None,59JFC302A1,5C2F309007,M2EMIX01203,2026-03-09 21:02:14,5CF3D1D4C1,M2ECOT001,2026-03-14 04:44:03,5CF3D1D4R1,M2EROL013,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,None,59JFC302A1,5C1F325056,M2EMIX01103,2026-03-25 10:17:47,5CF3P182C1,M2ECOT001,2026-03-25 22:57:45,5CF3P182R1,M2EROL012,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14824,None,59JFC302A1,5A4F319058,M2EMIX01802,2026-03-19 14:19:24,5AF3J185C1,M2ECOT002,2026-03-20 01:20:13,5AF3J185R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14825,None,59JFC302A1,5A4F320064,M2EMIX01802,2026-03-20 06:51:07,5AF3K141C1,M2ECOT002,2026-03-20 14:53:16,5AF3K141R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14826,None,59JFC302A1,5A4F320064,M2EMIX01802,2026-03-20 06:51:07,5AF3K177C1,M2ECOT002,2026-03-20 23:44:37,5AF3K177R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14827,None,59JFC302A1,5A2F318044,M2EMIX01602,2026-03-18 07:28:19,5AF3I188C1,M2ECOT002,2026-03-19 01:53:33,5AF3I188R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
idx = 7
filename = f'feature_store_v10_n32s_{idx}.parquet'
data_chunk = pd.read_parquet(filename)


for col in [x for x in data_chunk.columns if 'Date' in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))

for col in [x for x in data_chunk.columns if 'ID' in x and 'Degas' not in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))
display(data_chunk)

if data is None : 
    data = data_chunk
else :
    data = pd.concat([data, data_chunk], axis=0)
    #data = pd.concat([data, data_chunk], axis=0, ignore_index=True, sort=False)
    del data_chunk
    gc.collect()

datetime64[ns]
object
object
object
object
datetime64[ns]
datetime64[ns]
float64
float64
object
object
object
object
object
object
object
object
object
object
object
object
object
object
object
float64
float64
float64
float64
object
object


,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,None,59JFC302A1,5A2F318044,M2EMIX01602,2026-03-18 07:28:19,5AF3I188C1,M2ECOT002,2026-03-19 01:53:33,5AF3I188R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,None,59JFC302A1,5A2F318044,M2EMIX01602,2026-03-18 07:28:19,5AF3I188C1,M2ECOT002,2026-03-19 01:53:33,5AF3I188R1,M2EROL011,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,None,59JFC302A1,5A4F320064,M2EMIX01802,2026-03-20 06:51:07,5AF3K141C1,M2ECOT002,2026-03-20 14:53:16,5AF3K141R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,None,59JFC302A1,5A4F320064,M2EMIX01802,2026-03-20 06:51:07,5AF3K141C1,M2ECOT002,2026-03-20 14:53:16,5AF3K141R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,None,59JFC302A1,5A4F320064,M2EMIX01802,2026-03-20 06:51:07,5AF3K182C1,M2ECOT002,2026-03-21 00:41:39,5AF3K182R1,M2EROL011,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14824,None,59JFC302A1,5A4F320066,M2EMIX01802,2026-03-20 15:59:56,5AF3K177C1,M2ECOT002,2026-03-20 23:44:37,5AF3K177R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14825,None,59JFC302A1,5C1F324051,M2EMIX01103,2026-03-24 15:31:54,5CF3P182C1,M2ECOT001,2026-03-25 22:57:45,5CF3P182R1,M2EROL012,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14826,None,59JFC302A1,5C2F324047,M2EMIX01203,2026-03-24 12:58:56,5CF3P135C1,M2ECOT001,2026-03-25 14:02:37,5CF3P135R1,M2EROL015,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14827,None,59JFC302A1,5C2F324047,M2EMIX01203,2026-03-24 12:58:56,5CF3P164C1,M2ECOT001,2026-03-25 19:24:35,5CF3P164R1,M2EROL012,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
data

,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,07TCED7LGC0021G2B2016718,59JFB112A1,5A2F201001,M2EMIX01602,2026-02-02 01:28:10,5CF1T1A5C1,M2ECOT001,2026-01-30 06:12:56,5CF1T1A5R2,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07TCED7LGC0021G2E2063322,59JFB142A1,5A2F201001,M2EMIX01602,2026-02-02 01:28:10,5AF2A131C1,M2ECOT002,2026-02-10 14:14:12,5AF2A131R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,07TCED7LGC0021G2G2018702,59JFB162A1,5A3F209041,M2EMIX01702,2026-02-09 19:11:46,5CF1V165C1,M2ECOT001,2026-01-31 18:56:59,5CF1V165R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07TCED7LGC0021G382040158,59JFC082A1,5C1F225029,M2EMIX01103,2026-02-26 00:00:57,5CF2R151C1,M2ECOT001,2026-02-27 17:44:29,5CF2R151R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07TCED7LGC0021G3S2044730,59JFC252A1,5C2F309007,M2EMIX01203,2026-03-09 21:02:14,5CF3E183C1,M2ECOT001,2026-03-14 20:58:31,5CF3E183R1,M2EROL015,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14824,None,59JFC302A1,5A4F320066,M2EMIX01802,2026-03-20 15:59:56,5AF3K177C1,M2ECOT002,2026-03-20 23:44:37,5AF3K177R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14825,None,59JFC302A1,5C1F324051,M2EMIX01103,2026-03-24 15:31:54,5CF3P182C1,M2ECOT001,2026-03-25 22:57:45,5CF3P182R1,M2EROL012,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14826,None,59JFC302A1,5C2F324047,M2EMIX01203,2026-03-24 12:58:56,5CF3P135C1,M2ECOT001,2026-03-25 14:02:37,5CF3P135R1,M2EROL015,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14827,None,59JFC302A1,5C2F324047,M2EMIX01203,2026-03-24 12:58:56,5CF3P164C1,M2ECOT001,2026-03-25 19:24:35,5CF3P164R1,M2EROL012,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
data['01_Mixing_Lot ID'] = data['01_Mixing_Lot ID'].astype(str)
data['01_Mixing_Equipment ID'] = data['01_Mixing_Equipment ID'].astype(str)

In [18]:
data['01_Mixing_Finished Date'] = data['01_Mixing_Finished Date'].astype(str)
data = data[data['01_Mixing_Finished Date']!="[]"]

In [19]:
def tmp2(x) :
    if x == "[]" :
        y = ""
    else : 
        y = x
    return y

In [20]:
for col_name in [x for x in data.columns if 'Date' in x] :
    data[col_name] = data[col_name].astype(str)

    data[col_name] = data[col_name].apply(lambda x : tmp2(x))   

In [21]:
for col_name in [x for x in data.columns if 'Lot ID' in x] :
    data[col_name] = data[col_name].astype(str)

    data[col_name] = data[col_name].apply(lambda x : tmp2(x))   

In [22]:
for col_name in [x for x in data.columns if 'Equipment ID' in x] :
    data[col_name] = data[col_name].astype(str)

    data[col_name] = data[col_name].apply(lambda x : tmp2(x))   

In [23]:
for col_name in [x for x in data.columns if 'Cell ID' in x] :
    data[col_name] = data[col_name].astype(str)

    data[col_name] = data[col_name].apply(lambda x : tmp2(x))   

In [24]:
data.to_parquet('feature_store_v10_n32s.parquet')

In [25]:
data

,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,07TCED7LGC0021G2B2016718,59JFB112A1,5A2F201001,M2EMIX01602,2026-02-02 01:28:10,5CF1T1A5C1,M2ECOT001,2026-01-30 06:12:56,5CF1T1A5R2,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07TCED7LGC0021G2E2063322,59JFB142A1,5A2F201001,M2EMIX01602,2026-02-02 01:28:10,5AF2A131C1,M2ECOT002,2026-02-10 14:14:12,5AF2A131R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,07TCED7LGC0021G2G2018702,59JFB162A1,5A3F209041,M2EMIX01702,2026-02-09 19:11:46,5CF1V165C1,M2ECOT001,2026-01-31 18:56:59,5CF1V165R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07TCED7LGC0021G382040158,59JFC082A1,5C1F225029,M2EMIX01103,2026-02-26 00:00:57,5CF2R151C1,M2ECOT001,2026-02-27 17:44:29,5CF2R151R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07TCED7LGC0021G3S2044730,59JFC252A1,5C2F309007,M2EMIX01203,2026-03-09 21:02:14,5CF3E183C1,M2ECOT001,2026-03-14 20:58:31,5CF3E183R1,M2EROL015,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14824,None,59JFC302A1,5A4F320066,M2EMIX01802,2026-03-20 15:59:56,5AF3K177C1,M2ECOT002,2026-03-20 23:44:37,5AF3K177R1,M2EROL014,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14825,None,59JFC302A1,5C1F324051,M2EMIX01103,2026-03-24 15:31:54,5CF3P182C1,M2ECOT001,2026-03-25 22:57:45,5CF3P182R1,M2EROL012,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14826,None,59JFC302A1,5C2F324047,M2EMIX01203,2026-03-24 12:58:56,5CF3P135C1,M2ECOT001,2026-03-25 14:02:37,5CF3P135R1,M2EROL015,...,162.5,NaN,234.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14827,None,59JFC302A1,5C2F324047,M2EMIX01203,2026-03-24 12:58:56,5CF3P164C1,M2ECOT001,2026-03-25 19:24:35,5CF3P164R1,M2EROL012,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
